In [1]:
import os
import numpy as np
from PIL import Image

import random
import torch
import torch.nn.functional as F
from torchvision.utils import save_image
import torch.optim as optim

from torchvision import transforms
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

from diffusers import StableDiffusionInpaintPipeline, AutoencoderKL
import timm
import lpips
import sys

# sys.path.append('/root/StableGuard/locmark') # instead of the top directory
from locmark.helper import load_images_from_path, norm_imagenet, denorm_imagenet
from locmark.locmark import LocMark

val_transforms = transforms.Compose([
    transforms.Resize((256,256)),
    # transforms.CenterCrop(224),
    transforms.ToTensor(),
    # normalize_img,
])

file_name = '0034.png'

def load_img(path, transforms=None):
    img = Image.open(path).convert("RGB")
    img = transforms(img).unsqueeze(0).to(device)
    return img

def norm_tensor(tensor):
    t = tensor.clone().detach()
    
    min_val = t.min()
    max_val = t.max()

    tensor_norm = (tensor - min_val) / (max_val - min_val)

    print(f"Tensor normalized: min={tensor_norm.min()}, max={tensor_norm.max()}")
    
    return tensor_norm, min_val, max_val

def denorm_tensor(tensor, original_min=None, original_max=None):
    t = tensor.clone().detach()

    return t * (original_max - original_min) + original_min

def create_random_mask(img_pt, num_masks=1, mask_percentage=0.1, max_attempts=100):
    _, _, height, width = img_pt.shape
    mask_area = int(height * width * mask_percentage)
    masks = torch.zeros((num_masks, 1, height, width), dtype=img_pt.dtype)

    if mask_percentage >= 0.999:
        # Full mask for entire image
        return torch.ones((num_masks, 1, height, width), dtype=img_pt.dtype).to(img_pt.device)

    for ii in range(num_masks):
        placed = False
        attempts = 0
        while not placed and attempts < max_attempts:
            attempts += 1

            max_dim = int(mask_area ** 0.5)
            mask_width = random.randint(1, max_dim)
            mask_height = mask_area // mask_width

            # Allow broader aspect ratios for larger masks
            aspect_ratio = mask_width / mask_height if mask_height != 0 else 0
            if 0.25 <= aspect_ratio <= 4:  # Looser ratio constraint
                if mask_height <= height and mask_width <= width:
                    x_start = random.randint(0, width - mask_width)
                    y_start = random.randint(0, height - mask_height)
                    overlap = False
                    for jj in range(ii):
                        if torch.sum(masks[jj, :, y_start:y_start + mask_height, x_start:x_start + mask_width]) > 0:
                            overlap = True
                            break
                    if not overlap:
                        masks[ii, :, y_start:y_start + mask_height, x_start:x_start + mask_width] = 1
                        placed = True

        if not placed:
            # Fallback: just fill a central region if all attempts fail
            print(f"Warning: Failed to place mask {ii}, using fallback.")
            center_h = height // 2
            center_w = width // 2
            half_area = int((mask_area // 2) ** 0.5)
            h_half = min(center_h, half_area)
            w_half = min(center_w, half_area)
            masks[ii, :, center_h - h_half:center_h + h_half, center_w - w_half:center_w + w_half] = 1

    return masks.to(img_pt.device)

/opt/conda/envs/stableguard/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class Params:
    """Hyperparameters and configuration settings for LocMark."""
    def __init__(self):
        # --- System & Paths ---
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.train_datasets = '/mnt/nas5/suhyeon/datasets/valAGE-Set'
        self.image_path = f'/mnt/nas5/suhyeon/datasets/valAGE-Set/{file_name}'
        self.exp_name = 'baseline'
        # self.output_dir = f'/mnt/nas5/suhyeon/projects/locmark/{self.exp_name}' # single image optimization
        # self.output_dir = f'/mnt/nas5/suhyeon/projects/eval_spliceless/ours' # NOTE: multi image optimization
        self.output_dir = "/mnt/nas5/suhyeon/projects/locmark/" # single image optimization
        self.single_image_mode = True # NOTE
        self.num_test_images = 100 # the first n images

        # --- Model Configurations ---
        self.vae_model_name = "stabilityai/stable-diffusion-2-1"
        self.vae_subfolder = "vae"
        
        # --- Image Size Parameters ---
        self.vae_image_size = 512
        self.image_size = 256
        self.transform = transforms.Compose([
            transforms.Resize((self.image_size, self.image_size)),
            transforms.ToTensor(),
        ])

        # --- LocMark Core Parameters ---
        self.margin = 1.0
        self.grid_size = 28
        self.mask_percentage = 0.3
        self.num_masks = 1
        self.seed = 42
        self.num_inference_steps = 100
        self.guidance_scale = 7.5
        self.temperature = 5.0
        self.target_cossim = 0.1

        # --- Optimization Parameters ---
        self.lr = 2.0
        self.steps = 300
        self.lambda_p = 0.1 #0.05 #0.025
        self.lambda_i = 0.05 #0.01 #0.005
        self.feat_layer = 1

        # --- Robustness Parameters --- 
        self.eps0_std = [0.0, 0.25] # Latent noise
        
        # --- Demo/Evaluation Parameters ---
        self.batch_size = 1
        # self.num_test_images = 1

        self.feature_dim = None
        # tiny, small
        if self.feat_layer == 0:
            self.feature_dim = 96
        elif self.feat_layer == 1:
            self.feature_dim = 192
        elif self.feat_layer == 2:
            self.feature_dim = 384
        elif self.feat_layer == 3:
            self.feature_dim = 768

In [3]:
def compute_psnr(a, b):
    mse = F.mse_loss(a, b).item()
    if mse == 0:
        return 100.0
    return 20 * torch.log10(1.0 / torch.sqrt(torch.tensor(mse)))

def calculate_iou(pred_mask, gt_mask):
    # Ensure masks are binary
    # pred_mask_bin = (pred_mask < 0).float()
    # pred_mask_bin = torch.sigmoid(pred_mask)
    # pred_mask_bin = (pred_mask_bin > 0.65).float() # Thresholding at 0.65
    # gt_mask_bin = (gt_mask > 0).float() # Ground truth might not be 0/1

    pred_mask_bin = pred_mask.float()
    gt_mask_bin = gt_mask.float()

    # save_image(pred_mask_bin * gt_mask_bin, "results/intersection.png")
    # save_image(pred_mask_bin + gt_mask_bin, "results/union.png")

    # Intersection and Union
    intersection = (pred_mask_bin * gt_mask_bin).sum()
    union = (pred_mask_bin + gt_mask_bin).sum() - intersection

    iou = intersection / (union + 1e-6) # Add epsilon to avoid division by zero
    return iou.item()

In [4]:
# img_path = "/mnt/nas5/suhyeon/projects/freq-loc/secret_code/0002.png"
# img_path = "/mnt/nas5/suhyeon/projects/freq-loc/secret_code/analysis_dist_wm_step400.png"
# img_path = "/mnt/nas5/suhyeon/projects/freq-loc/baseline/20251120-152728/watermarked/0003.png"
seed = 45
proportion_masked = 0.3
trials = 5
img_path = f"/mnt/nas5/suhyeon/projects/locmark/20251205-184848/watermarked/{file_name}"

In [5]:
pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "sd-legacy/stable-diffusion-inpainting",
    # torch_dtype=torch.float16,
    cache_dir='/mnt/nas5/suhyeon/caches'
).to(device)

args = Params()
locmark = LocMark(args=args)

# secret_key = torch.load('./learned_directional_vector.pt')
# locmark.direction_vectors = torch.tensor(secret_key).to(args.device)
# print(locmark.direction_vectors)

torch.manual_seed(seed)
generator = torch.Generator(device=device).manual_seed(seed)
to_tensor = transforms.ToTensor()

watermarked = load_img(img_path, transforms=args.transform)
original = load_img(f'/mnt/nas5/suhyeon/datasets/valAGE-Set/{file_name}', transforms=val_transforms)

original = F.interpolate(original, size=(512, 512), mode="bilinear", align_corners=False)
watermarked = F.interpolate(watermarked, size=(512, 512), mode="bilinear", align_corners=False)

psnrs = []
ious_sp = []
ious_sl = []
logits = []

for _ in range(trials):
    mask = create_random_mask(watermarked, num_masks=1, mask_percentage=proportion_masked)

    img_norm, min_norm, max_norm = norm_tensor(watermarked)
    img_edit_pil = pipe(prompt="", image=img_norm, mask_image=mask, generator=generator).images[0]
    img_edit = to_tensor(img_edit_pil)
    img_edit = img_edit.unsqueeze(0).to(device)

    img_edit = denorm_tensor(img_edit, min_norm, max_norm)  # [1, 3, H, W]
    img_edit_spliced = img_edit * mask + watermarked * (1-mask)

    img_edit = F.interpolate(img_edit, size=(args.image_size, args.image_size), mode="bilinear", align_corners=False)
    img_edit_spliced = F.interpolate(img_edit_spliced, size=(args.image_size, args.image_size), mode="bilinear", align_corners=False)
    decoded_batch = locmark.decode_watermark(img_edit)
    decoded_batch_spliced = locmark.decode_watermark(img_edit_spliced)

    save_image(img_edit, "results/edited_sl.png")
    save_image(img_edit_spliced, "results/edited_sp.png")

    original = F.interpolate(original, size=(args.image_size, args.image_size), mode="bilinear", align_corners=False)
    watermarked_224 = F.interpolate(watermarked, size=(args.image_size, args.image_size), mode="bilinear", align_corners=False)
    mask_224 = F.interpolate(mask, size=(args.image_size, args.image_size), mode="bilinear", align_corners=False)

    save_image(decoded_batch.float(), "results/pred_sl.png")
    save_image(decoded_batch_spliced.float(), "results/pred_sp.png")
    save_image((1-mask_224).float(), "results/gt.png")

    psnrs.append(compute_psnr(watermarked_224, original))
    ious_sl.append(calculate_iou(decoded_batch, 1-mask_224))
    ious_sp.append(calculate_iou(decoded_batch_spliced, 1-mask_224))
    logits.append(decoded_batch)
 
    print(f"PSNR: {psnrs[-1]:.2f}, IoU_Spliced: {ious_sp[-1]:.4f} IoU_Spliceless: {ious_sl[-1]:.4f}")

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]An error occurred while trying to fetch /mnt/nas5/suhyeon/caches/models--sd-legacy--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet: Error no file named diffusion_pytorch_model.safetensors found in directory /mnt/nas5/suhyeon/caches/models--sd-legacy--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
Loading pipeline components...:  71%|███████▏  | 5/7 [00:02<00:00,  2.63it/s]An error occurred while trying to fetch /mnt/nas5/suhyeon/caches/models--sd-legacy--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /mnt/nas5/suhyeon/caches/models--sd-legacy--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae.
Defaulting to unsafe 

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/opt/conda/envs/stableguard/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/envs/stableguard/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /opt/conda/envs/stableguard/lib/python3.12/site-packages/lpips/weights/v0.1/alex.pth
Tensor normalized: min=0.0, max=1.0


100%|██████████| 50/50 [00:08<00:00,  6.07it/s]


...

PSNR: 46.23, IoU_Spliced: 0.8513 IoU_Spliceless: 0.6975
Tensor normalized: min=0.0, max=1.0


100%|██████████| 50/50 [00:08<00:00,  6.03it/s]


PSNR: 46.23, IoU_Spliced: 0.8572 IoU_Spliceless: 0.7412
Tensor normalized: min=0.0, max=1.0


100%|██████████| 50/50 [00:08<00:00,  6.03it/s]


PSNR: 46.23, IoU_Spliced: 0.8838 IoU_Spliceless: 0.7733


In [6]:
print(f"## Average on {trials} trials ##")
print(f"PSNR (imperceptibility): {np.mean(psnrs):.2f} dB")
print(f"IoU (spliced): {np.mean(ious_sp):.4f}")
print(f"IoU (spliceless): {np.mean(ious_sl):.4f}")

## Average on 5 trials ##
PSNR (imperceptibility): 46.23 dB
IoU (spliced): 0.8640
IoU (spliceless): 0.7628


In [7]:
# sig = torch.sigmoid(torch.cat(logits, dim=0)).cpu().numpy().flatten()

In [8]:
# import matplotlib.pyplot as plt
# plt.figure(figsize=(10, 6))
# plt.hist(sig, bins=50, alpha=0.7)#, label='A: w/ L1 loss')
# plt.title('Logit Distribution Comparison')
# plt.xlabel('Logit Value')
# plt.ylabel('Frequency')
# plt.legend()
# plt.grid(True)
# plt.savefig('logits_comparison.png')
# # print("\nSaved logit distribution histogram to 'logit_histogram.png'")

In [9]:
# # logits_a = torch.load("logits_wo_loss.pt").cpu().numpy().flatten()
# logits_a = torch.load("logits_wo_loss.pt").cpu().numpy().flatten()
# logits_b = torch.load("logits_w_l1_loss.pt").cpu().numpy().flatten()
# logits_c = total_logits
# print(f"[A: w/o Add. Loss]Mean: {logits_a.mean():.2f}, Std: {logits_a.std():.2f}, Min: {logits_a.min():.2f}, Max: {logits_a.max():.2f}")
# print(f"[B: w/ L1 Loss] Mean: {logits_b.mean():.2f}, Std: {logits_b.std():.2f}, Min: {logits_b.min():.2f}, Max: {logits_b.max():.2f}")
# print(f"[B: w/ L1 Loss (Dual)] Mean: {logits_c.mean():.2f}, Std: {logits_c.std():.2f}, Min: {logits_c.min():.2f}, Max: {logits_c.max():.2f}")

# import matplotlib.pyplot as plt
# plt.figure(figsize=(10, 6))
# plt.hist(logits_a, bins=50, alpha=0.4, label='A: w/o L')
# plt.hist(logits_b, bins=50, alpha=0.4, label='B: w/ L')
# plt.hist(logits_c, bins=50, alpha=0.4, label='B: w/ L (Dual)')
# plt.title('Logit Distribution Comparison')
# plt.xlabel('Logit Value')
# plt.ylabel('Frequency')
# plt.legend()
# plt.grid(True)
# plt.savefig('logits_comparison.png')


In [10]:
# sig_a = torch.sigmoid(torch.load("logits_wo_loss.pt")).cpu().numpy().flatten()
# sig_b = torch.sigmoid(torch.load("logits_w_l1_loss.pt")).cpu().numpy().flatten()
# sig_c = torch.sigmoid(torch.load("logits_w_l1_loss_dual.pt")).cpu().numpy().flatten()
# print(f"[A: w/o Add. Loss]Mean: {sig_a.mean():.2f}, Std: {sig_a.std():.2f}, Min: {sig_a.min():.2f}, Max: {sig_a.max():.2f}")
# print(f"[B: w/ L1 Loss] Mean: {sig_b.mean():.2f}, Std: {sig_b.std():.2f}, Min: {sig_b.min():.2f}, Max: {sig_b.max():.2f}")
# print(f"[C: w/ L1 Loss (Dual)] Mean: {sig_c.mean():.2f}, Std: {sig_c.std():.2f}, Min: {sig_c.min():.2f}, Max: {sig_c.max():.2f}")

<!--  -->

In [11]:
# import matplotlib.pyplot as plt
# plt.figure(figsize=(10, 6))
# plt.hist(sig_a, bins=50, alpha=0.7, label='A: w/o Add. Loss')
# plt.hist(sig_b, bins=50, alpha=0.7, label='B: w/ L1 Loss')
# plt.hist(sig_c, bins=50, alpha=0.7, label='B: w/ L1 Loss (Dual)')
# plt.title('Logit Distribution Comparison')
# plt.xlabel('Logit Value')
# plt.ylabel('Frequency')
# plt.legend()
# plt.grid(True)
# plt.savefig('logits_comparison.png')
# # print("\nSaved logit distribution histogram to 'logit_histogram.png'")

In [12]:
# 